<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-04/NB04_chinese_room.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 04: The Chinese Room and Its Replies

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-04/lab.html>.

## Overview

Searle's Chinese Room is the most discussed argument against strong AI. This week reconstructs the argument, examines the main replies and Searle's answers, and asks what the argument shows about present language models [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to reconstruct the Chinese Room argument and its later axiomatic form, to explain the systems, robot and brain simulator replies and Searle's answers, to assess the luminous room objection, and to state precisely which conclusion the argument supports.

## The argument

Searle imagined himself locked in a room with a large rulebook written in English [1]. Batches of Chinese characters are passed in, and by following the rules, which relate shapes to shapes, he passes back other Chinese characters. The rules are so good that people outside, who read the answers, cannot tell them from those of a native speaker. Yet Searle understands no Chinese: He manipulates symbols by their shapes alone. Since a computer running a program does nothing more than Searle does in the room, running a program cannot be sufficient for understanding. The target is strong AI, not the usefulness of AI.

A decade later, Searle presented the reasoning as an argument from axioms [2]. Programs are formal, or syntactic. Minds have mental contents, or semantics. Syntax by itself is neither constitutive of nor sufficient for semantics. Therefore programs are neither constitutive of nor sufficient for minds. The lab of this week puts students inside the room and then asks them to test their position on these axioms.

## The replies

Searle discussed several replies in the original article [1]. The systems reply grants that the man does not understand but holds that the whole system, man, rulebook and papers together, does. Searle answered that he could memorise the rules and do everything in his head and would still understand nothing. The robot reply proposes putting the program in a robot with cameras and motors, so that its symbols are connected to the world. Searle answered that the new inputs are just more symbols to him. The brain simulator reply proposes a program that simulates the firing of neurons in a Chinese speaker's brain. Searle answered that simulating the formal structure of neuron firings misses the brain's causal powers, and he illustrated the point with a system of water pipes and valves.

Churchland and Churchland argued that the third axiom is not self-evident but an empirical claim, and they offered an analogy [4]. A person waving a magnet in a dark room produces electromagnetic waves but no visible light; it would be a mistake to conclude that light is not electromagnetic radiation. Intuitions about slow and small cases can mislead about fast and large ones. Preston and Bishop collected later essays that continue the debate from many directions [3].

## What the argument shows and what it does not

Precision about the conclusion matters. The argument claims that running a program is not sufficient for understanding. It does not claim that machines cannot understand, since Searle held that a machine with causal powers equivalent to those of brains could think. It also leaves open what those causal powers are, which critics regard as a weakness. The argument relies on an intuition, that the man in the room does not understand, and on a step from the man to the system. Most replies attack that step or the reliability of the intuition, as Figure 4.1 shows. Harnad read the room as an illustration of a more specific problem, namely how symbols can be connected to what they mean, which is the topic of Week 5 [5].

![Figure 4.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-04/figures/w04_fig1.png)

*Figure 4.1. Searle's axiomatic argument and the points at which four replies attack it [2, 4].*

## The room and language models

Large language models are trained on text alone and produce fluent text by transforming symbols, so the argument seems to apply to them directly. There are also differences. No person wrote their rules; the rules are learned statistical regularities distributed over billions of parameters. Defenders of the systems reply argue that whatever understanding exists would belong to the trained system as a whole, not to any component. Critics answer that learning the rules does not change their syntactic character. The notebook builds two rooms, one that only stores question-answer pairs and one that composes answers from a small model of a world, and compares how they handle new questions. Both only manipulate symbols, but they differ in how they generalise, which is one way of making the debate empirical.

> **Pause and reflect.** In the lab, did following the rulebook feel like understanding? Does your answer tell you anything about a system that follows billions of rules at high speed?

# Hands-on lab

The notebook builds two rooms for a small symbolic world: one that stores question-answer pairs, like a pure rulebook, and one that composes answers from a model of the world. It measures how each handles questions it has never seen and computes the size of a complete rulebook [1, 6].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A small world and its questions

In [ ]:
import itertools
rng = np.random.default_rng(SEED)
OBJECTS = ["ball", "cup", "box", "book", "lamp", "shoe"]
COLOURS = ["red", "blue", "green", "yellow"]
PLACES = ["table", "shelf", "floor"]
WORLD = {o: (COLOURS[i % 4], PLACES[i % 3]) for i, o in enumerate(OBJECTS)}
QUESTIONS = [f"is the {o} {c} ?" for o in OBJECTS for c in COLOURS] + [f"is the {o} on the {p} ?" for o in OBJECTS for p in PLACES]

def truth(q):
    w = q.split()
    return "yes" if (w[3] == WORLD[w[2]][0] if len(w) == 5 else w[5] == WORLD[w[2]][1]) else "no"

order = rng.permutation(len(QUESTIONS))
train = [QUESTIONS[i] for i in order[:30]]
test = [QUESTIONS[i] for i in order[30:]]
print(len(QUESTIONS), "questions,", len(train), "in the rulebook,", len(test), "new")

## 2. Room 1: a pure rulebook

The rulebook pairs each stored question with its answer. For any other input it passes back a fixed note.

In [ ]:
RULEBOOK = {q: truth(q) for q in train}

def rulebook_room(q):
    return RULEBOOK.get(q, "i do not understand")

print(rulebook_room(train[0]), "|", rulebook_room(test[0]))

## 3. Room 2: composing answers from a world model

The second room parses the question and consults a table of facts. It still manipulates symbols only, but it composes answers.

In [ ]:
FACTS = {(o, "colour", WORLD[o][0]) for o in OBJECTS} | {(o, "place", WORLD[o][1]) for o in OBJECTS}

def model_room(q):
    w = q.split()
    fact = (w[2], "colour", w[3]) if len(w) == 5 else (w[2], "place", w[5])
    return "yes" if fact in FACTS else "no"

print(model_room(test[0]), "| true answer:", truth(test[0]))

### Your turn, exercise 1

Compute the share of the new questions (`test`) answered correctly by each room. Return a tuple `(rulebook_share, model_share)`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _coverage_reference():
    return (float(np.mean([rulebook_room(q) == truth(q) for q in test])), float(np.mean([model_room(q) == truth(q) for q in test])))

In [ ]:
shares = None  # your computation
check("Exercise 1", shares, _coverage_reference())

## 4. How large must a complete rulebook be?

For a vocabulary of V words, the number of word sequences of length 1 to n is V + V^2 + ... + V^n.

### Your turn, exercise 2

Compute the number of possible inputs of length 1 to 8 for a vocabulary of 50 words.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _rulebook_size_reference():
    return sum(50 ** k for k in range(1, 9))

In [ ]:
rulebook_size = None  # your computation
check("Exercise 2", rulebook_size, _rulebook_size_reference())
print("Write two sentences: Does the success of the model room support the systems reply, or is it still syntax only?")

## Visual exploration: How large would the rulebook be?

The left panel estimates the number of possible conversations that a Blockhead-style lookup table would have to cover. The right panel shows how much of a Zipf-distributed stream of questions a table of a given size answers.

In [ ]:
turns = np.arange(1, 11)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for V in [1000, 10000, 50000]:
    ax[0].plot(turns, turns * 20 * np.log10(V), "o-", label=f"vocabulary {V:,}")   # 20 words per turn
ax[0].axhline(80, color="k", ls="--", lw=0.8); ax[0].text(1, 84, "about 10^80 atoms in the observable universe", fontsize=8)
ax[0].set_xlabel("turns in the conversation"); ax[0].set_ylabel("log10 of possible conversations"); ax[0].legend()
ranks = np.arange(1, 1_000_001); pz = 1 / ranks ** 1.1; pz /= pz.sum()
sizes = [10 ** k for k in range(1, 7)]
ax[1].semilogx(sizes, [pz[:s].sum() for s in sizes], "o-")
ax[1].set_xlabel("entries in the rulebook"); ax[1].set_ylabel("share of questions answered"); ax[1].set_title("Coverage of a Zipf-distributed question stream")
plt.tight_layout(); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-04/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which reply do you find strongest, and what would Searle say in response? Write the exchange as a short dialogue.
2. Does the difference between the two rooms in the notebook matter for the argument? Why or why not?
3. Apply the argument to a language model assistant you use. Which premise would its defenders reject?

## Weekly task and submission

Write an argument analysis of about 600 words that reconstructs the Chinese Room argument in numbered form, evaluates the systems reply and the luminous room objection, and states whether the argument applies to present language models. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**The systems reply after four decades.** Survey the main defences and criticisms of the systems reply and decide whether the reply succeeds against the Chinese Room [1, 3].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Searle, J. R. (1980). Minds, brains, and programs. *Behavioral and Brain Sciences*, 3(3), 417-424. <https://doi.org/10.1017/S0140525X00005756>

[2] Searle, J. R. (1990). Is the brain's mind a computer program?. *Scientific American*, 262(1), 26-31. <https://doi.org/10.1038/scientificamerican0190-26>

[3] Preston, J., & Bishop, M. (Eds.) (2002). *Views into the Chinese Room: New Essays on Searle and Artificial Intelligence*. Clarendon Press.

[4] Churchland, P. M., & Churchland, P. S. (1990). Could a machine think?. *Scientific American*, 262(1), 32-37. <https://doi.org/10.1038/scientificamerican0190-32>

[5] Harnad, S. (1990). The symbol grounding problem. *Physica D: Nonlinear Phenomena*, 42(1-3), 335-346. <https://doi.org/10.1016/0167-2789(90)90087-6>

[6] Block, N. (1981). Psychologism and behaviorism. *The Philosophical Review*, 90(1), 5-43. <https://doi.org/10.2307/2184371>